# 04 - Training Model

Split waktu: train sampai Desember 2023, validasi 2024, test Januari 2025 – September 2026.
Seleksi model memakai **MAPE validasi**. Pilihan model vs harga bulan lalu dilakukan per komoditas, juga pada data validasi.

Notebook ini tidak menimpa model di `models/`. Untuk menyimpan model final, jalankan `python retrain.py`.


In [1]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

In [2]:
from src.data_loader import load_long
from src.feature_engineering import build_features
from src import training as T
feat = build_features(load_long())
train, val, test, res, pv, pt, store = T.run_experiments(feat)

Split -> train 66,737 (2016-10-01..2023-12-01) | val 9,925 (2024-01-01..2024-12-01) | test 27,798 (2025-01-01..2026-09-01)
  [level|global  |linear] {} val MAPE=5.21% (1s)
  [level|global  |rf    ] {'n_estimators': 100, 'max_depth': 12, 'min_samples_leaf': 10, 'max_features': 0.6} val MAPE=4.86% (10s)
  [level|global  |rf    ] {'n_estimators': 100, 'max_depth': 18, 'min_samples_leaf': 5, 'max_features': 0.6} val MAPE=4.85% (12s)
  [level|global  |xgb   ] {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8} val MAPE=4.76% (3s)
  [level|global  |xgb   ] {'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8} val MAPE=4.91% (4s)
  [level|global  |xgb   ] {'n_estimators': 600, 'max_depth': 8, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8} val MAPE=4.76% (10s)
  [level|separate|linear] {} val MAPE=5.74% (1s)
  [level|separate|rf    ] {'n_estimators': 100, 'max_depth': 18, 'm

In [3]:
res[['model','split','MAPE_%','MAE','RMSE','R2']].sort_values(['split','MAPE_%']).round(3)

,model,split,MAPE_%,MAE,RMSE,R2
25,Naive Forecast,test,4.912,1838.051,3725.780,0.903
15,Random Forest | global | ratio,test,5.175,1902.441,3622.975,0.908
17,XGBoost | global | ratio,test,5.415,1968.961,3651.175,0.907
13,Linear Regression | global | ratio,test,5.511,1979.929,3628.960,0.908
7,Linear Regression | separate | level,test,5.590,1989.237,3533.812,0.913
5,XGBoost | global | level,test,5.608,2023.728,3694.725,0.904
3,Random Forest | global | level,test,5.648,2002.967,3635.919,0.907
21,Random Forest | separate | ratio,test,5.731,2022.761,3707.155,0.904
19,Linear Regression | separate | ratio,test,5.805,2081.920,3694.946,0.904
23,XGBoost | separate | ratio,test,5.894,2056.068,3688.737,0.905


In [4]:
print('Model terpilih (MAPE validasi terendah):', T.select_best(res))

Model terpilih (MAPE validasi terendah): Random Forest | global | ratio


## Pilihan per komoditas: model vs harga bulan lalu

Dihitung dari prediksi validasi dan test, sama seperti di `retrain.py`. Test hanya dilaporkan, tidak dipakai untuk memilih.

In [5]:
from src.pipeline import commodity_choice
best = T.select_best(res)
choice = commodity_choice(val, test, pv[best], pt[best])
choice[['nama','metode','mape_val_model','mape_val_naive','mape_test_terpilih','band_low_pct','band_high_pct','cakupan_uji_pct']]

,nama,metode,mape_val_model,mape_val_naive,mape_test_terpilih,band_low_pct,band_high_pct,cakupan_uji_pct
0,Telur,naive,3.441,2.901,4.470,-5.455,4.000,64.39
1,Bawang Putih,naive,3.972,3.627,5.336,-4.762,8.725,66.17
2,Daging Ayam,naive,5.734,5.661,7.020,-6.739,13.208,72.69
3,Minyak Nabati,naive,1.810,1.440,1.148,-0.418,4.348,83.91
4,Bawang Merah,model,12.516,15.933,12.443,-14.694,24.074,77.82
5,Beras,naive,1.947,1.220,0.834,-1.724,1.852,82.61
6,Gula,naive,1.849,1.129,1.200,-2.618,2.813,80.60
